In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%bash
T=/content/drive/MyDrive/tse
P=/content/tse-project--1-/tse-project
if [ -f $T/code/train.py ]; then
  mkdir -p $P && cp -r $T/code/. $P/
else
  cd /content && git clone https://github.com/frecklehead/tse-project--1-.git
  sed -i 's/run_opts={"device": str(device)}/run_opts={"device": "cuda:0" if str(device) == "cuda" else str(device)}/' $P/tse/speaker.py
fi
cd $P && pip install -q -r requirements.txt
grep -n run_opts tse/speaker.py
mkdir -p data && cp $T/archive/*.jsonl $T/archive/*.pt data/
ls data

14:        run_opts={"device": "cuda:0" if str(device) == "cuda" else str(device)},
noise_test.jsonl
noise_train.jsonl
seen_emb.pt
seen_speech.jsonl
test_emb.pt
test_speech.jsonl
train_emb.pt
train_speech.jsonl
val_emb.pt
val_speech.jsonl


In [ ]:
%%bash
apt-get -qq install -y aria2 > /dev/null
D=/content/tse-project--1-/tse-project/data
mkdir -p $D/dl $D/DEMAND $D/LibriSpeech && cd $D/dl
declare -A N=([dev-clean]=40 [test-clean]=40 [train-clean-100]=251 [train-clean-360]=921)
for s in dev-clean test-clean train-clean-100 train-clean-360; do
  if [ "$(ls $D/LibriSpeech/$s 2>/dev/null | wc -l)" -eq "${N[$s]}" ]; then echo "$s present"; continue; fi
  aria2c -x 16 -s 16 -k 4M -c -q --max-tries=0 --retry-wait=5 https://www.openslr.org/resources/12/$s.tar.gz \
    && tar -xzf $s.tar.gz -C $D && rm $s.tar.gz
  echo "$s: $(ls $D/LibriSpeech/$s | wc -l) speakers (expected ${N[$s]})"
done
for n in DKITCHEN DLIVING DWASHING NFIELD NPARK NRIVER OHALLWAY OMEETING OOFFICE PCAFETER PRESTO PSTATION SPSQUARE STRAFFIC TBUS TCAR TMETRO; do
  if [ "$(find $D/DEMAND/$n -name '*.wav' 2>/dev/null | wc -l)" -ge 16 ]; then continue; fi
  for try in 1 2 3 4 5 6; do
    aria2c -x 2 -s 2 -c -q --max-tries=3 --retry-wait=10 https://zenodo.org/records/1227121/files/${n}_16k.zip && break
    echo "$n attempt $try failed, waiting 60s"; sleep 60
  done
  [ -f ${n}_16k.zip ] && unzip -q -o ${n}_16k.zip -d $D/DEMAND/$n && rm ${n}_16k.zip
  sleep 5
done
for d in $D/DEMAND/*/; do echo "$(basename $d): $(find $d -name '*.wav' | wc -l) wavs"; done
df -h /content | tail -1

dev-clean: 40 speakers (expected 40)
test-clean: 40 speakers (expected 40)
train-clean-100: 251 speakers (expected 251)
train-clean-360: 921 speakers (expected 921)
DKITCHEN: 16 wavs
DLIVING: 16 wavs
DWASHING: 16 wavs
NFIELD: 16 wavs
NPARK: 16 wavs
NRIVER: 16 wavs
OHALLWAY: 16 wavs
OMEETING: 16 wavs
OOFFICE: 16 wavs
PCAFETER: 16 wavs
PRESTO: 16 wavs
PSTATION: 16 wavs
SPSQUARE: 16 wavs
STRAFFIC: 16 wavs
TBUS: 16 wavs
TCAR: 16 wavs
TMETRO: 16 wavs
overlay         113G   75G   39G  67% /


In [ ]:
%%bash
T=/content/drive/MyDrive/tse
P=/content/tse-project--1-/tse-project; D=$P/data; cd $P
if [ -f $T/archive_v2/train2_emb.pt ]; then
  cp $T/archive_v2/* $D/ && echo "restored v2 manifests"
else
  python prepare_data.py speech --root $D --subdir LibriSpeech/train-clean-360 --out $D/train360_speech.jsonl --max-utts 40
  python precompute_embeddings.py --root $D --manifests $D/train360_speech.jsonl --out $D/train360_emb.pt
  python - <<'EOF'
import json, torch
D = '/content/tse-project--1-/tse-project/data'
rd = lambda f: [json.loads(l) for l in open(f'{D}/{f}')]
wr = lambda f, rows: open(f'{D}/{f}', 'w').write(''.join(json.dumps(r) + '\n' for r in rows))
rows = rd('train_speech.jsonl') + rd('train360_speech.jsonl')
wr('train2_speech.jsonl', rows)
a = torch.load(f'{D}/train_emb.pt'); a.update(torch.load(f'{D}/train360_emb.pt'))
torch.save(a, f'{D}/train2_emb.pt')
VAL = {'DKITCHEN', 'OOFFICE', 'TBUS'}          # noise categories used ONLY for validation
nz = rd('noise_train.jsonl')
wr('noise_train2.jsonl', [r for r in nz if r['category'] not in VAL])
wr('noise_val2.jsonl',   [r for r in nz if r['category'] in VAL])
print(len({r['speaker'] for r in rows}), 'speakers,', len(rows), 'utterances,', len(a), 'embeddings')
EOF
  mkdir -p $T/archive_v2
  cp $D/train2_speech.jsonl $D/train2_emb.pt $D/noise_train2.jsonl $D/noise_val2.jsonl $T/archive_v2/
fi
wc -l $D/train2_speech.jsonl $D/noise_train2.jsonl $D/noise_val2.jsonl

restored v2 manifests
  61318 /content/tse-project--1-/tse-project/data/train2_speech.jsonl
    160 /content/tse-project--1-/tse-project/data/noise_train2.jsonl
     48 /content/tse-project--1-/tse-project/data/noise_val2.jsonl
  61526 total


In [ ]:
%%writefile /content/tse-project--1-/tse-project/train_v2.py
"""train.py + failsafes: local-first checkpoints, fault-tolerant Drive mirror, rolling snapshots,
emergency save on SIGTERM/SIGINT, resume from the newest of last.pt / emergency.pt."""
import argparse, csv, json, os, shutil, signal, sys, time
import numpy as np
import torch
from torch.utils.data import DataLoader
from tse.data import STAGES, TSEDataset
from tse.losses import si_sdr, sisdr_loss
from tse.models import MODEL_KEYS, build_model, count_params

STOP = {"flag": False}


def get_args():
    p = argparse.ArgumentParser()
    p.add_argument("--data-root", required=True)
    p.add_argument("--train-speech", required=True)
    p.add_argument("--train-emb", required=True)
    p.add_argument("--train-noise", default=None)
    p.add_argument("--val-speech", required=True)
    p.add_argument("--val-emb", required=True)
    p.add_argument("--val-noise", default=None)
    p.add_argument("--out-dir", required=True)
    p.add_argument("--mirror-dir", default=None, help="e.g. a Drive folder; failures here never stop training")
    p.add_argument("--snapshot-every", type=int, default=5)
    p.add_argument("--patience", type=int, default=5, help="ReduceLROnPlateau patience")
    p.add_argument("--cond", default="film",
                   choices=["none", "concat", "film", "gating", "concat2", "film2", "gating2"])
    for k, d in dict(N=256, L=16, B=128, H=256, P=3, X=8, R=3).items():
        p.add_argument(f"--{k}", type=int, default=d)
    p.add_argument("--segment", type=float, default=3.0)
    p.add_argument("--batch", type=int, default=6)
    p.add_argument("--accum", type=int, default=1)
    p.add_argument("--epochs", type=int, default=60)
    p.add_argument("--steps-per-epoch", type=int, default=500)
    p.add_argument("--stage-epochs", type=int, default=8)
    p.add_argument("--val-size", type=int, default=200)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--workers", type=int, default=2)
    p.add_argument("--amp", action="store_true")
    p.add_argument("--max-hours", type=float, default=None)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--no-resume", action="store_true")
    return p.parse_args()


def save_atomic(state, path):
    tmp = path + ".tmp"
    torch.save(state, tmp)
    os.replace(tmp, path)


def mirror_file(src, mirror_dir, rel=None):
    """Copy src into mirror_dir; retry; never raise."""
    if not mirror_dir or not os.path.exists(src):
        return False
    rel = rel or os.path.basename(src)
    dst = os.path.join(mirror_dir, rel)
    for attempt in range(3):
        try:
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            tmp = dst + ".tmp"
            shutil.copyfile(src, tmp)
            os.replace(tmp, dst)
            return True
        except Exception as e:
            print(f"[warn] mirror {rel} failed ({attempt + 1}/3): {e}", flush=True)
            time.sleep(5)
    print(f"[warn] could not mirror {rel}; training continues, local copy is safe", flush=True)
    return False


def on_signal(signum, frame):
    STOP["flag"] = True
    print(f"[signal {signum}] stopping: will write emergency.pt", flush=True)


@torch.no_grad()
def validate(model, dl, device):
    model.eval()
    imps, abss = [], []
    for b in dl:
        mix, tgt, emb = b["mixture"].to(device), b["target"].to(device), b["emb"].to(device)
        est = model(mix, emb)
        s_est, s_mix = si_sdr(est, tgt), si_sdr(mix, tgt)
        imps.append((s_est - s_mix).cpu())
        abss.append(s_est.cpu())
    return torch.cat(imps).mean().item(), torch.cat(abss).mean().item()


def main():
    a = get_args()
    torch.manual_seed(a.seed)
    np.random.seed(a.seed)
    os.makedirs(a.out_dir, exist_ok=True)
    signal.signal(signal.SIGTERM, on_signal)
    signal.signal(signal.SIGINT, on_signal)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = a.amp and device.type == "cuda"
    print(f"device={device} amp={use_amp}", flush=True)

    cfg = {k: getattr(a, k) for k in MODEL_KEYS}
    model = build_model(cfg).to(device)
    print(f"model params: {count_params(model):,}  cfg={cfg}", flush=True)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=a.patience)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    train_ds = TSEDataset(a.data_root, a.train_speech, a.train_emb, a.train_noise,
                          segment_sec=a.segment, length=a.steps_per_epoch * a.batch)
    val_ds = TSEDataset(a.data_root, a.val_speech, a.val_emb, a.val_noise,
                        segment_sec=a.segment, length=a.val_size, deterministic=True, seed=12345)
    train_dl = DataLoader(train_ds, batch_size=a.batch, num_workers=a.workers, drop_last=True,
                          pin_memory=device.type == "cuda")
    val_dl = DataLoader(val_ds, batch_size=a.batch, num_workers=a.workers)

    P = {n: os.path.join(a.out_dir, n) for n in ("last.pt", "best.pt", "emergency.pt", "log.csv")}

    # fresh runtime: pull state back from the mirror (Drive)
    if a.mirror_dir and not a.no_resume and not os.path.exists(P["last.pt"]) and not os.path.exists(P["emergency.pt"]):
        for n, dst in P.items():
            src = os.path.join(a.mirror_dir, n)
            if os.path.exists(src):
                shutil.copyfile(src, dst)
                print(f"restored {n} from mirror", flush=True)

    start_epoch, best = 0, -1e9
    if not a.no_resume:
        cands = []
        for n in ("last.pt", "emergency.pt"):
            if os.path.exists(P[n]):
                try:
                    ck = torch.load(P[n], map_location=device, weights_only=False)
                    cands.append((ck.get("saved_at", 0), n, ck))
                except Exception as e:
                    print(f"[warn] could not read {n}: {e}", flush=True)
        if cands:
            _, n, ck = max(cands, key=lambda c: c[0])
            if ck["model_cfg"] != cfg:
                raise SystemExit("Checkpoint model config differs from current arguments. Use a new --out-dir.")
            model.load_state_dict(ck["model"])
            opt.load_state_dict(ck["opt"])
            sched.load_state_dict(ck["sched"])
            scaler.load_state_dict(ck["scaler"])
            start_epoch, best = ck["epoch"] + 1, ck["best"]
            print(f"RESUMED from {n}: continuing at epoch {start_epoch}, best val SI-SDRi={best:.2f}", flush=True)

    torch.manual_seed(a.seed + start_epoch)  # fresh random stream after each resume

    # keep the log consistent with the checkpoint we resumed from
    header = ["epoch", "stage", "train_loss", "val_sisdri", "val_sisdr", "lr", "minutes"]
    rows = []
    if os.path.exists(P["log.csv"]):
        with open(P["log.csv"]) as f:
            rows = [r for r in list(csv.reader(f))[1:] if r and int(r[0]) < start_epoch]
    with open(P["log.csv"], "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)

    def state(epoch):
        return {"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "scaler": scaler.state_dict(), "epoch": epoch, "best": best, "model_cfg": cfg,
                "saved_at": time.time()}

    t0 = time.time()
    for epoch in range(start_epoch, a.epochs):
        stage = min(len(STAGES) - 1, epoch // a.stage_epochs)
        train_ds.set_stage(stage)
        model.train()
        opt.zero_grad(set_to_none=True)
        tot, n, te = 0.0, 0, time.time()
        for step, b in enumerate(train_dl):
            if STOP["flag"]:
                break
            mix, tgt, emb = b["mixture"].to(device), b["target"].to(device), b["emb"].to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                est = model(mix, emb)
            loss = sisdr_loss(est, tgt)
            scaler.scale(loss / a.accum).backward()
            if (step + 1) % a.accum == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
            tot += loss.item()
            n += 1

        if STOP["flag"]:  # emergency save: weights are mid-epoch, bookkeeping says "epoch-1 finished"
            save_atomic(state(epoch - 1), P["emergency.pt"])
            mirror_file(P["emergency.pt"], a.mirror_dir)
            print("emergency.pt saved; exiting", flush=True)
            sys.exit(2)

        v_imp, v_abs = validate(model, val_dl, device)
        sched.step(v_imp)
        lr = opt.param_groups[0]["lr"]
        mins = (time.time() - te) / 60
        print(f"epoch {epoch:3d} stage {stage} train_loss {tot / max(n, 1):7.3f} "
              f"val SI-SDRi {v_imp:6.2f} dB (SI-SDR {v_abs:6.2f}) lr {lr:.1e} [{mins:.1f} min]", flush=True)

        improved = v_imp > best
        if improved:
            best = v_imp
            save_atomic({"model": model.state_dict(), "model_cfg": cfg, "epoch": epoch,
                         "val_sisdri": v_imp, "saved_at": time.time()}, P["best.pt"])
        save_atomic(state(epoch), P["last.pt"])
        with open(P["log.csv"], "a", newline="") as f:
            csv.writer(f).writerow([epoch, stage, f"{tot / max(n, 1):.4f}", f"{v_imp:.3f}",
                                    f"{v_abs:.3f}", lr, f"{mins:.2f}"])
        if improved:
            mirror_file(P["best.pt"], a.mirror_dir)
        mirror_file(P["last.pt"], a.mirror_dir)
        mirror_file(P["log.csv"], a.mirror_dir)

        if a.snapshot_every and (epoch + 1) % a.snapshot_every == 0:
            snap = os.path.join(a.out_dir, "snapshots", f"ep{epoch:03d}.pt")
            os.makedirs(os.path.dirname(snap), exist_ok=True)
            save_atomic({"model": model.state_dict(), "model_cfg": cfg, "epoch": epoch, "val_sisdri": v_imp}, snap)
            mirror_file(snap, a.mirror_dir, os.path.join("snapshots", f"ep{epoch:03d}.pt"))

        if a.max_hours and (time.time() - t0) / 3600 >= a.max_hours:
            print("--max-hours reached; stopped cleanly.", flush=True)
            break

    with open(os.path.join(a.out_dir, "args.json"), "w") as f:
        json.dump(vars(a), f, indent=2)
    mirror_file(os.path.join(a.out_dir, "args.json"), a.mirror_dir)


if __name__ == "__main__":
    main()

Overwriting /content/tse-project--1-/tse-project/train_v2.py


In [ ]:
import os, shutil
P = '/content/tse-project--1-/tse-project'
m = f'{P}/tse/models.py'; s = open(m).read()
if 'ConditionerV2' not in s:
    new_code = '''class ConditionerV2(nn.Module):
    """Unit-RMS embedding, identity at init, bounded scale. Modes: concat/film/gating."""

    def __init__(self, mode, C, E):
        super().__init__()
        if mode not in ("concat", "film", "gating"):
            raise ValueError(f"unknown conditioning mode: {mode}")
        self.mode, self.E, self.C = mode, E, C
        if mode == "concat":
            self.proj = nn.Conv1d(C + E, C, 1)
            with torch.no_grad():
                self.proj.weight.zero_()
                self.proj.weight[:, :C, 0] = torch.eye(C)  # pass features through, embedding weights start at 0
                self.proj.bias.zero_()
        else:
            self.fc = nn.Linear(E, 2 * C if mode == "film" else C)
            nn.init.zeros_(self.fc.weight)
            nn.init.zeros_(self.fc.bias)

    def forward(self, x, e):
        e = (F.normalize(e.float(), dim=-1) * (self.E ** 0.5)).to(x.dtype)
        if self.mode == "concat":
            et = e.unsqueeze(-1).expand(-1, -1, x.size(-1))
            return self.proj(torch.cat([x, et], dim=1))
        if self.mode == "film":
            g, b = self.fc(e).chunk(2, dim=-1)
            return x * (1 + torch.tanh(g)).unsqueeze(-1) + b.unsqueeze(-1)
        return x * (2 * torch.sigmoid(self.fc(e))).unsqueeze(-1)


def make_conditioner(mode, C, E):
    if mode.endswith("2"):
        return ConditionerV2(mode[:-1], C, E)
    return Conditioner(mode, C, E)


'''
    anchor = 'class TSEConvTasNet(nn.Module):'
    old = 'self.conds.append(Conditioner(cond, B, emb_dim))'
    assert anchor in s and old in s, 'models.py differs from the version I saw; stop and send it to me'
    s = s.replace(anchor, new_code + anchor, 1).replace(old, 'self.conds.append(make_conditioner(cond, B, emb_dim))', 1)
    open(m, 'w').write(s)
print('ConditionerV2 present:', 'ConditionerV2' in open(m).read())

ConditionerV2 present: True


In [ ]:
import sys, importlib, torch
sys.path.insert(0, '/content/tse-project--1-/tse-project')
import tse.models as M; importlib.reload(M)
for c in ['concat2', 'film2', 'gating2']:
    m = M.build_model(dict(N=64, L=16, B=32, H=64, P=3, X=4, R=2, cond=c)).eval()
    x = torch.randn(2, 16000); x2 = x.clone(); x2[:, 12000:] = torch.randn(2, 4000)   # change only the future
    with torch.no_grad():
        a = m(x, torch.randn(2, 192)); b = m(x, torch.randn(2, 192) * 5); c2 = m(x2, torch.zeros(2, 192) + 1)
    print(c, 'shape', tuple(a.shape),
          '| identity at init (~0):', f'{(a - b).abs().max().item():.1e}',
          '| causal (~0):', f'{(a[:, :10000] - c2[:, :10000]).abs().max().item():.1e}')

concat2 shape (2, 16000) | identity at init (~0): 0.0e+00 | causal (~0): 0.0e+00
film2 shape (2, 16000) | identity at init (~0): 0.0e+00 | causal (~0): 0.0e+00
gating2 shape (2, 16000) | identity at init (~0): 0.0e+00 | causal (~0): 0.0e+00


In [ ]:
import os, shutil
P = '/content/tse-project--1-/tse-project'
D = '/content/drive/MyDrive/tse/code'
skip = shutil.ignore_patterns('data', '__pycache__', '.git', 'runs', '*.pyc')
shutil.copytree(P, D, ignore=skip, dirs_exist_ok=True)

def has(path, text): return text in open(os.path.join(D, path)).read()
print('train.py present        :', os.path.exists(f'{D}/train.py'))
print('train_v2.py present     :', os.path.exists(f'{D}/train_v2.py'))
print('speaker.py cuda:0 patch :', has('tse/speaker.py', 'cuda:0'))
print('models.py ConditionerV2 :', has('tse/models.py', 'ConditionerV2'))
print('train_v2.py film2 option:', has('train_v2.py', '"film2"'))

train.py present        : True
train_v2.py present     : True
speaker.py cuda:0 patch : True
models.py ConditionerV2 : True
train_v2.py film2 option: True


In [ ]:
%%bash
cd /content/tse-project--1-/tse-project; D=data
rm -rf /content/runs/smoke
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train_v2.py --data-root $D --train-speech $D/train2_speech.jsonl --train-emb $D/train2_emb.pt --train-noise $D/noise_train2.jsonl \
  --val-speech $D/val_speech.jsonl --val-emb $D/val_emb.pt --val-noise $D/noise_val2.jsonl \
  --out-dir /content/runs/smoke --cond film2 --N 512 --H 512 --batch 3 --accum 2 --segment 3.0 \
  --epochs 1 --steps-per-epoch 100 --val-size 30 --amp --workers 2 2>&1 | grep -v Warning
nvidia-smi --query-gpu=memory.used,memory.total --format=csv

Process is terminated.


In [ ]:
import subprocess, time, csv, os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # reduces memory fragmentation

RUN, COND = 'film_v2', 'film2'
EPOCHS, STEPS = 80, 1600          # batch 3 x accum 2 = effective batch 6; 1600 micro-batches = 800 updates per epoch
                                  # for a relay across accounts, set EPOCHS to this session's stopping point (e.g. 27)

P = '/content/tse-project--1-/tse-project'; D = P + '/data'; os.chdir(P)
OUT, MIRROR = f'/content/runs/{RUN}', f'/content/drive/MyDrive/tse/runs/{RUN}'
CMD = ['python', 'train_v2.py', '--data-root', D,
       '--train-speech', f'{D}/train2_speech.jsonl', '--train-emb', f'{D}/train2_emb.pt', '--train-noise', f'{D}/noise_train2.jsonl',
       '--val-speech', f'{D}/val_speech.jsonl', '--val-emb', f'{D}/val_emb.pt', '--val-noise', f'{D}/noise_val2.jsonl',
       '--out-dir', OUT, '--mirror-dir', MIRROR, '--cond', COND,
       '--N', '512', '--H', '512', '--batch', '3', '--accum', '2', '--segment', '3.0',
       '--epochs', str(EPOCHS), '--steps-per-epoch', str(STEPS), '--stage-epochs', '10',
       '--val-size', '200', '--patience', '5', '--lr', '0.001', '--amp', '--workers', '2', '--snapshot-every', '5']

def last_epoch():
    try:
        rows = list(csv.reader(open(f'{OUT}/log.csv')))[1:]
        return int(rows[-1][0]) if rows else -1
    except Exception:
        return -1

fails = 0
while True:
    before = last_epoch()
    rc = subprocess.call(CMD)
    after = last_epoch()
    if rc == 0 and after >= EPOCHS - 1:
        print('TRAINING COMPLETE'); break
    fails = 0 if after > before else fails + 1
    if fails >= 4:
        print('giving up after 4 attempts with no progress; check the error above'); break
    print(f'exit code {rc}, last epoch {after}; restarting in 30 s'); time.sleep(30)

In [ ]:
import os, sys, numpy as np, soundfile as sf, torch
P = '/content/tse-project--1-/tse-project'; D = P + '/data'; os.chdir(P); sys.path.insert(0, P)
from tse.data import TSEDataset
from tse.models import build_model
from tse.losses import si_sdr

CKPT = '/content/runs/film_v2/best.pt'                        # or last.pt for the newest weights
if not os.path.exists(CKPT): CKPT = '/content/drive/MyDrive/tse/runs/film_v2/best.pt'
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
ck = torch.load(CKPT, map_location='cpu', weights_only=False)
model = build_model(ck['model_cfg']); model.load_state_dict(ck['model']); model.to(dev).eval()
print('checkpoint:', CKPT, '| epoch', ck.get('epoch'), '| its val SI-SDRi', ck.get('val_sisdri'))

SETS = {
    'train': (f'{D}/train_speech.jsonl', f'{D}/train_emb.pt', f'{D}/noise_train2.jsonl'),   # voices and noise the model trained on
    'new':   (f'{D}/test_speech.jsonl',  f'{D}/test_emb.pt',  f'{D}/noise_test.jsonl'),      # unseen speakers and unseen noise
}
N = 20
norm = lambda x: x / (np.abs(x).max() + 1e-8) * 0.9
for tag, (sp, em, nz) in SETS.items():
    ds = TSEDataset(D, sp, em, nz, segment_sec=3.0, length=N, deterministic=True, seed=7, p_no_interferer=0.0)
    imps = []
    for k in range(N):
        b = ds[k]
        mix = b['mixture'].unsqueeze(0).to(dev); tgt = b['target'].unsqueeze(0).to(dev); emb = b['emb'].unsqueeze(0).to(dev)
        with torch.no_grad():
            est = model(mix, emb)
        imps.append((si_sdr(est, tgt) - si_sdr(mix, tgt)).item())
        if k == 0:
            for name, x in (('mix', mix), ('out', est), ('target', tgt)):
                sf.write(f'/content/check_{tag}_{name}.wav', norm(x[0].cpu().numpy()), 16000)
    print(f'{tag:>5}: mean SI-SDRi {np.mean(imps):5.2f} dB | std {np.std(imps):4.2f} | first clip {imps[0]:5.2f} dB')

checkpoint: /content/runs/film_v2/best.pt | epoch 21 | its val SI-SDRi 5.902973175048828
train: mean SI-SDRi  3.88 dB | std 3.25 | first clip  3.62 dB
  new: mean SI-SDRi  3.60 dB | std 4.16 | first clip 11.32 dB
